# 02 — Deploy pipelines

Wealth ML SageMaker Foundation (WSF).

Use the **output of notebook 01**, not its metadata CSV.

| Mode | Data input here | What the pipeline's GetData step does |
|---|---|---|
| csv_direct | Generated sample.csv (or another ready flat CSV) | Reads the mounted CSV and copies it unchanged to raw/data.csv |
| feature_store | Feature Store configuration URI printed by notebook 01 | Queries the existing store and writes raw/data.csv |

The feature mapping is used for Feature Store ingestion/query configuration;
CSV-direct does not require it. Both routes hand a flat CSV to FeatureSelection.
The modes share an output contract, not necessarily identical column sets:
the current Feature Store query selects configured model features, while
CSV-direct preserves all supplied columns.

Paste your values into the first two input cells. The third cell contains
**Defaults**, including namespace and data_source.
This notebook publishes **one shared training pipeline and one shared inference pipeline**.
Adding or renaming a model folder updates their ModelId dropdown when republished.
The default names are **wsf-dev-training** and **wsf-dev-inference**.
They use `wsf-dev/` in S3 and leave the existing `source-dev` pipelines and outputs in place.
It does not create one pipeline pair per model.

Use a fresh kernel for this checkout; an older checkout can otherwise remain imported.

For initial testing, keep `registry_mode="offline_registry"` and `UseHPO=false`.
After training, supply its exact `model/application/model.tar.gz` URI as
**ModelDataUri** when starting inference with the same **ModelId**.

## Your AWS values

- Paste your bucket, pipeline role and existing container image together into this cell.
- These three values are required for either data-source mode.
- Presets such as namespace and data_source are in the separate Defaults cell below.

In [ ]:
bucket = ""
role_arn = ""
image_uri = ""

## Your data input — copy the block from notebook 01

- In csv_direct mode, csv_input is the generated sample.csv path or an exact S3 CSV URI; leave the Feature Store URI empty.
- In feature_store mode, supply the configuration URI printed after ingestion; leave csv_input empty.
- Do not put the table metadata CSV here. Notebook 01 uses that file to generate the sample.
- Each model's training target comes from its own parameters.json; it is not another notebook input.

In [ ]:
csv_input = ""
feature_store_config_uri = ""

## Defaults — change only when needed

- Keep namespace="wsf-dev" to publish the new pipeline pair alongside source-dev.
- Keep data_source="csv_direct" or change it to feature_store to match notebook 01.
- Registry mode, model selection and job settings are grouped here.
- CSV preparation caching is enabled; these defaults do not reset your pasted AWS or data inputs.

In [ ]:
namespace = "wsf-dev"
data_source = "csv_direct"
environment = "dev"               # Registry suffix: dev or prod.
registry_mode = "offline_registry" # Or model_registry once platform groups/permissions are ready.
deploy_name = "my-review"
model_ids = None                   # Discover all model folders; or select a list of folder names.
inference_model_uri = ""          # Optional default exact model.tar.gz; normally set when starting inference.
cache_preparation = True           # Reuse CSV preparation for 24 hours; Train still runs.
universe_data_root = ""            # Optional population/target root for Feature Store mode.
job_role_arn = ""                  # Empty uses role_arn for jobs as well as pipelines.
batch_runtime = "processing"
infrastructure = {
    "network": {"isolation": False, "subnet_ids": [], "security_group_ids": []},
    "output_kms_key_arn": "",
    "volume_kms_key_arn": "",
}
csv_preflight_ok = csv_upload_verified = False
print("Input mode:", data_source, "| cache_preparation:", cache_preparation)

## Validate inputs and source

- Read `label` from each selected model's `parameters.json` and show the model-to-target mapping.
- Stop on a missing CSV, missing target, duplicate header or Python syntax error.
- Preview the actual input and record its checksum.
- Discover folders containing parameters.json and show the output root.

In [ ]:
from pathlib import Path
from uuid import uuid4
import csv
import hashlib
import io
import json
import sys
import boto3
import pandas as pd

csv_preflight_ok = csv_upload_verified = False
bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
csv_input = csv_input.strip()
print("Effective csv_input:", repr(csv_input))
print("Current notebook directory:", Path.cwd())
print("cache_preparation:", cache_preparation)
assert bucket and "/" not in bucket and role_arn and image_uri, "STOP: fill in bucket, role_arn and image_uri."
assert data_source in {"feature_store", "csv_direct"}, "STOP: choose csv_direct or feature_store."
if data_source == "feature_store":
    assert feature_store_config_uri, "STOP: supply the existing Feature Store configuration URI."
else:
    assert csv_input, "STOP: csv_input is empty. Copy the sample.csv path from notebook 01. Nothing will be published."

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models"), str(ROOT / "mlops-models/src"), str(ROOT / "mlops-features")]
import mlops_models
import mlops_skeleton
for package in (mlops_models, mlops_skeleton):
    assert Path(package.__file__).resolve().is_relative_to(ROOT.resolve()), (
        f"STOP: this kernel loaded code from {package.__file__}. Restart this notebook's kernel for the new checkout."
    )
from mlops_models.pipelines.source import (
    source_identity, source_location, upload_source, build_definitions, preparation_fingerprint,
)
from mlops_skeleton.names import discover_models, validate_namespace

validate_namespace(namespace)
available_models = [folder.name for folder in discover_models(ROOT / "mlops-models/src/mlops_models/models")]
model_ids = available_models if model_ids is None else model_ids
assert model_ids and all(model in available_models for model in model_ids), f"STOP: select model folders from {available_models}."
def csv_settings():
    # Re-read target settings so later cells cannot use a stale CSV check after a model edit.
    return csv_input, tuple(
        (model, json.loads((ROOT / "mlops-models/src/mlops_models/models" / model / "parameters.json").read_text())["label"])
        for model in model_ids
    )

model_targets = {}
for model in model_ids:
    parameters_path = ROOT / "mlops-models/src/mlops_models/models" / model / "parameters.json"
    parameters = json.loads(parameters_path.read_text())
    target = parameters.get("label")
    assert isinstance(target, str) and target.strip(), f"STOP: set label in {parameters_path} to this model's target column."
    model_targets[model] = target
    print("Model:", model, "| Target:", repr(target), "| Parameters:", parameters_path)

for code_file in (ROOT / "mlops-models/src/mlops_models").rglob("*.py"):
    compile(code_file.read_text(), str(code_file), "exec")
print("Model Python syntax checks passed:", model_ids)
pipeline_role_arn = role_arn
job_role_arn = job_role_arn.strip() or role_arn
session = boto3.Session()
sm = session.client("sagemaker")
s3_root = f"s3://{bucket}/{namespace}"
identity = source_identity("direct-deploy", deploy_name)
source_uri = source_location(s3_root, identity)
csv_input_uri, csv_local_path, csv_check = "", None, None
if data_source == "csv_direct":
    if csv_input.startswith("s3://"):
        csv_input_uri = csv_input
    else:
        csv_local_path = Path(csv_input).expanduser()
        if not csv_local_path.is_absolute():
            csv_local_path = ROOT / csv_local_path
        csv_local_path = csv_local_path.resolve()
        print("Resolved local CSV:", csv_local_path)
        assert csv_local_path.is_file(), f"STOP: CSV file does not exist: {csv_local_path}. Nothing will be published."
    try:
        if csv_local_path is not None:
            csv_bytes = csv_local_path.read_bytes()
            csv_input_uri = f"{s3_root}/inputs/csv-direct/{deploy_name}/data.csv"
        else:
            csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
            response = session.client("s3").get_object(Bucket=csv_bucket, Key=csv_key)
            with response["Body"] as stream:
                csv_bytes = stream.read()
    except Exception as error:
        raise RuntimeError(f"STOP: cannot read the requested CSV {csv_local_path or csv_input_uri}. Nothing will be published: {error}") from error
    columns = next(csv.reader(io.StringIO(csv_bytes.decode("utf-8-sig"))), [])
    print("CHECKED CSV:", str(csv_local_path or csv_input_uri))
    print("Bytes:", len(csv_bytes), "| Columns:", len(columns))
    print("First 30 column names:", columns[:30])
    print("Model targets from parameters.json:", model_targets)
    assert columns and all(columns), "STOP: CSV has no valid header."
    assert len(set(columns)) == len(columns), "STOP: CSV contains duplicate column names."
    required_targets = list(dict.fromkeys(model_targets.values()))
    missing = [name for name in required_targets if name not in columns]
    similar = [name for name in columns if name.casefold() in {target.casefold() for target in missing}]
    assert not missing, f"STOP: required target column(s) missing: {missing}. Case-only matches: {similar}. The shared CSV must contain the selected models' targets. Check csv_input and each model's parameters.json label; the notebook does not create or rename targets."
    preview = pd.read_csv(io.BytesIO(csv_bytes), encoding="utf-8-sig", nrows=5, keep_default_na=False)
    assert len(preview), "STOP: CSV has a header but no data rows."
    display(preview.iloc[:, :10])
    csv_check = {"source": str(csv_local_path or csv_input_uri), "sha256": hashlib.sha256(csv_bytes).hexdigest(),
                 "bytes": len(csv_bytes), "columns": columns, "required_targets": required_targets, "model_targets": model_targets}
    csv_checked_settings = csv_settings()
    csv_preflight_ok = True
    print("CSV CHECK PASSED | target(s):", required_targets, "| SHA-256:", csv_check["sha256"])
    print("Will use S3 CSV:", csv_input_uri)
    del csv_bytes

print("Checkout:", ROOT)
print("Region:", session.region_name)
print("Pipeline role:", pipeline_role_arn)
print("Job role:", job_role_arn)
print("Models:", model_ids)
print("Source upload destination:", source_uri)
print("Input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)
print("Pipelines:", namespace + "-training", "|", namespace + "-inference")
print("Run outputs:", s3_root + "/models/<ModelId>/runs/<execution-id>/")
print("Source identity:", identity)

## Pin the image

- Resolve a supplied image tag to its immutable digest.
- If only a repository is supplied, explicitly report the newest tagged image.
- Reuse that digest for the published definitions.

In [ ]:
if "@sha256:" not in image_uri:
    registry, repository = image_uri.split("/", 1)
    repository, _, tag = repository.partition(":")
    ecr = session.client("ecr")
    if tag:
        detail = ecr.describe_images(
            registryId=registry.split(".")[0], repositoryName=repository,
            imageIds=[{"imageTag": tag}],
        )["imageDetails"][0]
    else:
        # Repository address only: use its newest tagged image,.
        images = []
        for page in ecr.get_paginator("describe_images").paginate(
                registryId=registry.split(".")[0], repositoryName=repository,
                filter={"tagStatus": "TAGGED"}):
            images.extend(page["imageDetails"])
        assert images, "That repository has no tagged images; paste a full image URI instead."
        detail = max(images, key=lambda item: item["imagePushedAt"])
        print("Pinned the newest tagged image:", detail.get("imageTags"))
    image_uri = registry + "/" + repository + "@" + detail["imageDigest"]
print("Image:", image_uri)

## Build the shared pipelines

- Build one training definition and one inference definition.
- Populate both ModelId dropdowns with the selected folders.
- Save definitions locally; display cache settings and Registry group names.

In [ ]:
if data_source == "csv_direct":
    assert csv_preflight_ok and csv_checked_settings == csv_settings(), "STOP: rerun the CSV check after changing the input, selected models or their targets."

preparation_revision = ""
if cache_preparation and data_source == "csv_direct":
    preparation_revision = preparation_fingerprint(session, csv_input_uri, csv_local_path)
print("Preparation cache:", "enabled for 24 hours" if preparation_revision else "disabled")
if preparation_revision:
    print("Preparation revision:", preparation_revision)
    print("GetData, FeatureSelection and Preprocess may reuse prior successful outputs; Train runs.")

definitions = build_definitions(
    namespace, job_role_arn, image_uri, s3_root, source_uri, identity,
    region=session.region_name, models=model_ids, infrastructure=infrastructure, batch_runtime=batch_runtime,
    feature_store_config_uri=feature_store_config_uri, universe_data_root=universe_data_root,
    data_source=data_source, csv_input_uri=csv_input_uri,
    preparation_cache_revision=preparation_revision, inference_model_uri=inference_model_uri,
    environment=environment, registry_mode=registry_mode,
)
WORK = ROOT / ".runs/source-deploy" / namespace
WORK.mkdir(parents=True, exist_ok=True)
for kind, body in definitions.items():
    path = WORK / (kind + ".json")
    path.write_text(json.dumps(body, indent=2) + "\n")
    print(namespace + "-" + kind, "→", " → ".join(step["Name"] for step in body["Steps"]))
    print("Definition:", path.resolve())
print("Registry mode:", registry_mode)
print("Registry groups:", [f"{model}-sm-{environment}" for model in model_ids])

print("Batch inference runtime:", batch_runtime)
print("UseHPO defaults to false; validate the image's tuning startup before enabling it.")

## Upload verified input and source

- Upload the selected local CSV, or verify the supplied S3 CSV.
- Compare the uploaded bytes with the preflight checksum.
- Upload the runtime source used by the jobs.

In [ ]:
csv_upload_verified = False
if data_source == "csv_direct":
    assert csv_preflight_ok and csv_checked_settings == csv_settings(), "STOP: rerun the CSV check after changing the input, selected models or their targets."
    csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
    try:
        if csv_local_path is not None:
            assert hashlib.sha256(csv_local_path.read_bytes()).hexdigest() == csv_check["sha256"], "STOP: local CSV changed after validation; rerun the notebook."
            kms = infrastructure.get("output_kms_key_arn", "")
            encryption = {"ServerSideEncryption": "aws:kms", "SSEKMSKeyId": kms} if kms else {}
            session.client("s3").upload_file(str(csv_local_path), csv_bucket, csv_key, ExtraArgs=encryption)
        response = session.client("s3").get_object(Bucket=csv_bucket, Key=csv_key)
        with response["Body"] as stream:
            uploaded_sha256 = hashlib.sha256(stream.read()).hexdigest()
        assert uploaded_sha256 == csv_check["sha256"], "S3 CSV differs from the file checked above."
    except Exception as error:
        raise RuntimeError(f"STOP: CSV upload/readback verification failed at {csv_input_uri}. Pipelines will not be published: {error}") from error
    csv_upload_verified = True
    print("S3 CSV VERIFIED:", csv_input_uri)
    print("SHA-256:", uploaded_sha256, "| Required target(s):", csv_check["required_targets"])
uploaded_source = upload_source(session, s3_root, identity)
print("Uploaded source:", uploaded_source)
print("GetData / batch Preprocess input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)

## Publish in Studio

- Create or update the same namespace-training and namespace-inference pipelines.
- Verify their saved definitions and roles; no execution is started.
- Save the publication record and print the ModelId choices.

In [ ]:
if data_source == "csv_direct":
    assert csv_upload_verified and csv_checked_settings == csv_settings(), "STOP: complete CSV verification before publishing."
    for body in definitions.values():
        saved_csv = next(p["DefaultValue"] for p in body["Parameters"] if p["Name"] == "CsvInputUri")
        assert saved_csv == csv_input_uri, "STOP: definition uses another CSV; rerun all cells."
published = {}
for kind, body in definitions.items():
    name = f"{namespace}-{kind}"
    request = {
        "PipelineName": name, "RoleArn": pipeline_role_arn,
        "PipelineDefinition": json.dumps(body),
    }
    try:
        existing = sm.describe_pipeline(PipelineName=name)
    except sm.exceptions.ClientError as error:
        if "does not exist" not in error.response["Error"]["Message"]:
            raise
        response = sm.create_pipeline(**request, Tags=[
            {"Key": "Deployment", "Value": namespace},
            {"Key": "SourceDeploy", "Value": identity["mode"]},
        ])
        version = 1  # The initial version of a newly created pipeline.
    else:
        tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=existing["PipelineArn"])["Tags"]}
        if tags.get("Deployment") != namespace or tags.get("SourceDeploy") != identity["mode"]:
            raise ValueError(f"{name} belongs to another deployment.")
        response = sm.update_pipeline(**request)
        version = response["PipelineVersionId"]
    actual = sm.describe_pipeline(PipelineName=name, PipelineVersionId=version)
    assert json.loads(actual["PipelineDefinition"]) == body, "Published definition differs; inspect before executing."
    assert actual["RoleArn"] == pipeline_role_arn, "Saved pipeline role differs; inspect before executing."
    published[kind] = {"name": name, "arn": response["PipelineArn"], "version": version}
    print("Published:", name, "| version:", version, "| saved role:", actual["RoleArn"])

publication = {
    "namespace": namespace, "pipelines": published, "models": model_ids,
    "source": identity, "source_uri": source_uri, "image": image_uri,
    "s3_root": s3_root, "batch_runtime": batch_runtime, "region": session.region_name,
    "environment": environment, "registry_mode": registry_mode,
    "feature_store_config_uri": feature_store_config_uri, "universe_data_root": universe_data_root,
    "data_source": data_source, "csv_input_uri": csv_input_uri,
    "preparation_cache_revision": preparation_revision,
    "csv_check": csv_check,
}
path = WORK / "publication.json"
path.write_text(json.dumps(publication, indent=2) + "\n")
print("Publication complete:", path.resolve())
print("One training pipeline and one inference pipeline are ready in Studio.")
print("ModelId dropdown:", model_ids)
if registry_mode == "offline_registry":
    print("Training skips Registry. After training, copy its model/application/model.tar.gz S3 URI.")
    print("Start the inference pipeline with the same ModelId and with ModelDataUri set to this exact artifact.")
    print("CsvInputUri selects scoring data. ModelDataUri selects the saved model.")
if data_source == "csv_direct":
    print("Use the newly printed pipeline version and this CsvInputUri:", csv_input_uri)
    print("A previous execution/version or a Studio parameter override can select a different CSV.")

## Execute

In SageMaker Studio, open **Pipelines** and select the printed training pipeline.
Create an execution, choose **ModelId**, keep **UseHPO=false**, and check **CsvInputUri**.
In offline mode keep **RegisterModel=false**.

After Train succeeds, copy the model URI from
`<s3_root>/models/<ModelId>/runs/<training-execution-id>/model/application/model.tar.gz`.
Start the inference pipeline with the same ModelId, that **ModelDataUri**, and the
scoring **CsvInputUri**. Final results are under the inference execution's
`predictions/predictions.csv`.

The CSV checked at publication is the shared training default, so it must contain
all selected models' target columns. Each model takes its own target from
`parameters.json` (`label` in this generic example). There is no notebook target
override. Update the demo parameter values when porting your model code; the
notebook does not invent labels or change CSV headers.

To switch models, start a new execution with another ModelId. Set CsvInputUri too
if that model uses different data; execution-time overrides need their own input validation. To add/rename models, edit the folders, restart
the notebook kernel, and run this notebook again. The containing checkout folder may have a different name;
run the notebook from inside that checkout so its ROOT discovery selects these files.